In [2]:
from vision import MyVision
from operate import Operator
op = Operator()

v = MyVision(yolo_model_path="models/best.pt")
image_path1 = "./tasks/transport/caini/00.png"
image_path2 = "./tasks/page-states/lingdi.png"

image_path2 = op.capture()
limit_scope = v.limit_scope(image_path1, scale=1.0)
scope = v.find_image(image_path2, image_path2, limit_scope)
scope
#op.click(scope)


[[1930.0, 1608.0], [3850.0, 2688.0]]

In [4]:
import win32gui
from mss import mss
import ctypes

# 1. 获取窗口句柄
hwnd = win32gui.FindWindow(None, "幸福小渔村")
if hwnd == 0:
    raise Exception("未找到目标窗口")

# 2. 获取客户区坐标（排除标题栏/边框）
left, top, right, bot = win32gui.GetClientRect(hwnd)
win32gui.ClientToScreen(hwnd, (left, top))  # 转换为屏幕绝对坐标

# 3. 处理DPI缩放，获取物理像素坐标
dpi = ctypes.windll.user32.GetDpiForWindow(hwnd)
scale = dpi / 96.0  # 96为基准DPI
# 将逻辑坐标乘以缩放因子，得到物理像素坐标
physical_left = int(left * scale)
physical_top = int(top * scale)
physical_width = int((right - left) * scale)
physical_height = int((bot - top) * scale)

# 4. 使用mss捕获物理像素区域
with mss() as sct:
    monitor = {
        "left": physical_left,
        "top": physical_top,
        "width": physical_width,
        "height": physical_height
    }
    sct_img = sct.grab(monitor)
    # 转换为PIL Image并保存
    from PIL import Image
    img = Image.frombytes("RGB", sct_img.size, sct_img.rgb)
    img.save("high_res_source.png")  # 此时获得的可能是接近2x,2y的清晰图


In [ ]:
import ctypes
ctypes.windll.shcore.SetProcessDpiAwareness(2)

import win32gui
import win32con
import win32process
import time


class WindowManager:
    def __init__(self, title):
        self.title = title
        self.hwnd = self._find_window(title)
    
    def _find_window(self, title):
        """查找窗口（支持模糊匹配）"""
        result = []
        
        def callback(hwnd, _):
            if win32gui.IsWindowVisible(hwnd):
                name = win32gui.GetWindowText(hwnd)
                if title in name:
                    result.append((hwnd, name))
        
        win32gui.EnumWindows(callback, None)
        
        if not result:
            raise Exception(f"❌ 未找到窗口: {title}")
        
        hwnd, name = result[0]
        print(f"✅ 找到窗口: {name} (句柄: {hwnd})")
        return hwnd
    
    # ---------- 弹到前台 ----------
    def activate(self):
        """激活窗口（弹到前台）"""
        if win32gui.IsIconic(self.hwnd):
            win32gui.ShowWindow(self.hwnd, win32con.SW_RESTORE)
        
        # 强制置前
        fg_hwnd = win32gui.GetForegroundWindow()
        fg_thread = ctypes.windll.user32.GetWindowThreadProcessId(fg_hwnd, None)
        my_thread = ctypes.windll.user32.GetWindowThreadProcessId(self.hwnd, None)
        
        if fg_thread != my_thread:
            ctypes.windll.user32.AttachThreadInput(fg_thread, my_thread, True)
        
        win32gui.SetForegroundWindow(self.hwnd)
        win32gui.BringWindowToTop(self.hwnd)
        
        if fg_thread != my_thread:
            ctypes.windll.user32.AttachThreadInput(fg_thread, my_thread, False)
        
        print("🔝 窗口已激活")
    
    # ---------- 置顶/取消 ----------
    def set_topmost(self, enable=True):
        """设置/取消始终置顶"""
        flag = win32con.HWND_TOPMOST if enable else win32con.HWND_NOTOPMOST
        win32gui.SetWindowPos(
            self.hwnd, flag, 0, 0, 0, 0,
            win32con.SWP_NOMOVE | win32con.SWP_NOSIZE
        )
        print(f"📌 置顶: {'开' if enable else '关'}")

    # ---------- 最小化 ----------
    def minimize(self):
        win32gui.ShowWindow(self.hwnd, win32con.SW_MINIMIZE)
        print("⬇️ 已最小化")

    # ---------- 最大化 ----------
    def maximize(self):
        win32gui.ShowWindow(self.hwnd, win32con.SW_MAXIMIZE)
        print("⬆️ 已最大化")

    # ---------- 恢复 ----------
    def restore(self):
        win32gui.ShowWindow(self.hwnd, win32con.SW_RESTORE)
        print("↩️ 已恢复")

    # ---------- 移动窗口 ----------
    def move(self, x, y, width=None, height=None):
        """移动窗口到指定位置"""
        if width is None or height is None:
            rect = win32gui.GetWindowRect(self.hwnd)
            width = width or (rect[2] - rect[0])
            height = height or (rect[3] - rect[1])
        win32gui.MoveWindow(self.hwnd, x, y, width, height, True)
        print(f"📐 移动到: ({x}, {y}) 大小: {width}x{height}")

    # ---------- 获取信息 ----------
    def get_info(self):
        rect = win32gui.GetWindowRect(self.hwnd)
        client = win32gui.GetClientRect(self.hwnd)
        return {
            "标题": win32gui.GetWindowText(self.hwnd),
            "句柄": self.hwnd,
            "窗口区域": rect,
            "客户区大小": (client[2], client[3]),
            "是否最小化": win32gui.IsIconic(self.hwnd),
            "是否可见": win32gui.IsWindowVisible(self.hwnd),
        }


# ================== 使用示例 ==================

wm = WindowManager("幸福小渔村")

# 查看窗口信息
info = wm.get_info()
for k, v in info.items():
    print(f"  {k}: {v}")

# 弹到前台
wm.activate()

# 始终置顶（可选）
# wm.set_topmost(True)

# 移动到屏幕左上角（可选）
# wm.move(0, 0)

# 最小化
# wm.minimize()

# 恢复
# wm.restore()

截图尺寸: (956, 524, 3)


True

In [ ]:
import ctypes
ctypes.windll.shcore.SetProcessDpiAwareness(2)

import win32gui
import win32con
import win32process
import time


class WindowManager:
    def __init__(self, title):
        self.title = title
        self.hwnd = self._find_window(title)
    
    def _find_window(self, title):
        """查找窗口（支持模糊匹配）"""
        result = []
        
        def callback(hwnd, _):
            if win32gui.IsWindowVisible(hwnd):
                name = win32gui.GetWindowText(hwnd)
                if title in name:
                    result.append((hwnd, name))
        
        win32gui.EnumWindows(callback, None)
        
        if not result:
            raise Exception(f"❌ 未找到窗口: {title}")
        
        hwnd, name = result[0]
        print(f"✅ 找到窗口: {name} (句柄: {hwnd})")
        return hwnd
    
    # ---------- 弹到前台 ----------
    def activate(self):
        """激活窗口（弹到前台）"""
        if win32gui.IsIconic(self.hwnd):
            win32gui.ShowWindow(self.hwnd, win32con.SW_RESTORE)
        
        # 强制置前
        fg_hwnd = win32gui.GetForegroundWindow()
        fg_thread = ctypes.windll.user32.GetWindowThreadProcessId(fg_hwnd, None)
        my_thread = ctypes.windll.user32.GetWindowThreadProcessId(self.hwnd, None)
        
        if fg_thread != my_thread:
            ctypes.windll.user32.AttachThreadInput(fg_thread, my_thread, True)
        
        win32gui.SetForegroundWindow(self.hwnd)
        win32gui.BringWindowToTop(self.hwnd)
        
        if fg_thread != my_thread:
            ctypes.windll.user32.AttachThreadInput(fg_thread, my_thread, False)
        
        print("🔝 窗口已激活")
    
    # ---------- 置顶/取消 ----------
    def set_topmost(self, enable=True):
        """设置/取消始终置顶"""
        flag = win32con.HWND_TOPMOST if enable else win32con.HWND_NOTOPMOST
        win32gui.SetWindowPos(
            self.hwnd, flag, 0, 0, 0, 0,
            win32con.SWP_NOMOVE | win32con.SWP_NOSIZE
        )
        print(f"📌 置顶: {'开' if enable else '关'}")

    # ---------- 最小化 ----------
    def minimize(self):
        win32gui.ShowWindow(self.hwnd, win32con.SW_MINIMIZE)
        print("⬇️ 已最小化")

    # ---------- 最大化 ----------
    def maximize(self):
        win32gui.ShowWindow(self.hwnd, win32con.SW_MAXIMIZE)
        print("⬆️ 已最大化")

    # ---------- 恢复 ----------
    def restore(self):
        win32gui.ShowWindow(self.hwnd, win32con.SW_RESTORE)
        print("↩️ 已恢复")

    # ---------- 移动窗口 ----------
    def move(self, x, y, width=None, height=None):
        """移动窗口到指定位置"""
        if width is None or height is None:
            rect = win32gui.GetWindowRect(self.hwnd)
            width = width or (rect[2] - rect[0])
            height = height or (rect[3] - rect[1])
        win32gui.MoveWindow(self.hwnd, x, y, width, height, True)
        print(f"📐 移动到: ({x}, {y}) 大小: {width}x{height}")

    # ---------- 获取信息 ----------
    def get_info(self):
        rect = win32gui.GetWindowRect(self.hwnd)
        client = win32gui.GetClientRect(self.hwnd)
        return {
            "标题": win32gui.GetWindowText(self.hwnd),
            "句柄": self.hwnd,
            "窗口区域": rect,
            "客户区大小": (client[2], client[3]),
            "是否最小化": win32gui.IsIconic(self.hwnd),
            "是否可见": win32gui.IsWindowVisible(self.hwnd),
        }


# ================== 使用示例 ==================

wm = WindowManager("幸福小渔村")

# 查看窗口信息
info = wm.get_info()
for k, v in info.items():
    print(f"  {k}: {v}")

# 弹到前台
wm.activate()

# 始终置顶（可选）
# wm.set_topmost(True)

# 移动到屏幕左上角（可选）
# wm.move(0, 0)

# 最小化
# wm.minimize()

# 恢复
#wm.restore()

✅ 找到窗口: 幸福小渔村 (句柄: 2622998)
  标题: 幸福小渔村
  句柄: 2622998
  窗口区域: (5, 8, 552, 1003)
  客户区大小: (531, 987)
  是否最小化: 0
  是否可见: 1
🔝 窗口已激活
↩️ 已恢复
